# Tech Challenge - Fase 2
## Integração com LLM para Interpretação dos Resultados

Neste notebook será integrada uma Large Language Model (LLM) ao sistema de diagnóstico desenvolvido nas etapas anteriores.

O objetivo é transformar informações técnicas do modelo, como probabilidade de malignidade, threshold utilizado e principais variáveis identificadas pelo SHAP, em uma explicação clara e estruturada para profissionais de saúde.

A LLM não será utilizada para realizar o diagnóstico. Sua função será exclusivamente interpretar e apresentar os resultados produzidos pelo modelo de Machine Learning.

In [1]:
from pathlib import Path

import json
import joblib
import numpy as np
import pandas as pd

print(
    "Bibliotecas carregadas com sucesso."
)

Bibliotecas carregadas com sucesso.


In [2]:
def localizar_arquivo(caminhos):

    for caminho in caminhos:

        if caminho.exists():
            return caminho

    raise FileNotFoundError(
        "Arquivo não encontrado."
    )


pasta_atual = Path.cwd()


caminho_modelo = localizar_arquivo([
    Path(
        "/app/models/"
        "modelo_ga_fase2.joblib"
    ),
    pasta_atual.parent
    / "models"
    / "modelo_ga_fase2.joblib"
])


caminho_threshold = localizar_arquivo([
    Path(
        "/app/results/"
        "threshold_otimizado.json"
    ),
    pasta_atual.parent
    / "results"
    / "threshold_otimizado.json"
])


caminho_shap = localizar_arquivo([
    Path(
        "/app/results/"
        "importancia_variaveis_shap.csv"
    ),
    pasta_atual.parent
    / "results"
    / "importancia_variaveis_shap.csv"
])


print(
    "Arquivos localizados com sucesso."
)

Arquivos localizados com sucesso.


In [3]:
modelo_ga = joblib.load(
    caminho_modelo
)


with open(
    caminho_threshold,
    "r",
    encoding="utf-8"
) as arquivo:

    configuracao_threshold = (
        json.load(
            arquivo
        )
    )


importancia_shap = pd.read_csv(
    caminho_shap
)


threshold_otimizado = (
    configuracao_threshold[
        "threshold_otimizado"
    ]
)


print(
    f"Threshold otimizado: "
    f"{threshold_otimizado:.2f}"
)

print()

print(
    "Principais variáveis SHAP:"
)

display(
    importancia_shap.head(10)
)

Threshold otimizado: 0.34

Principais variáveis SHAP:


,variavel,importancia_media_shap
0,texture_worst,0.543792
1,radius_worst,0.403651
2,concave points_worst,0.361740
3,perimeter_worst,0.358403
4,area_worst,0.352808
5,texture_mean,0.342873
6,concave points_mean,0.339866
7,symmetry_worst,0.315495
8,radius_se,0.300078
9,radius_mean,0.288084


## Preparação do contexto para o LLM

Para gerar uma explicação útil, o LLM receberá informações produzidas pelo modelo de Machine Learning.

Para cada caso serão utilizados:

- probabilidade estimada de malignidade;
- threshold utilizado na classificação;
- classificação final do modelo;
- principais variáveis que influenciaram a previsão segundo o SHAP.

O LLM não realizará um novo diagnóstico. Sua função será interpretar e apresentar, em linguagem natural, os resultados produzidos pelo modelo.

In [4]:
import shap

from sklearn.model_selection import train_test_split


caminho_dados = localizar_arquivo([
    Path("/app/data/data.csv"),
    pasta_atual / "data" / "data.csv",
    pasta_atual.parent / "data" / "data.csv"
])


dados = pd.read_csv(
    caminho_dados
)

df = dados.copy()

df = df.drop(
    columns=[
        "id",
        "Unnamed: 32"
    ],
    errors="ignore"
)

df["diagnosis"] = df["diagnosis"].map({
    "B": 0,
    "M": 1
})

df["diagnosis"] = (
    df["diagnosis"].astype(int)
)

df = (
    df.drop_duplicates()
    .reset_index(drop=True)
)

X = df.drop(
    columns=["diagnosis"]
)

y = df["diagnosis"]


X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )
)

print(
    "Dados preparados com sucesso."
)

print(
    f"Treino: {len(X_train)}"
)

print(
    f"Teste: {len(X_test)}"
)

Dados preparados com sucesso.
Treino: 455
Teste: 114


In [5]:
scaler = (
    modelo_ga.named_steps[
        "padronizacao"
    ]
)

regressao = (
    modelo_ga.named_steps[
        "modelo"
    ]
)


X_train_padronizado = pd.DataFrame(
    scaler.transform(
        X_train
    ),
    columns=X_train.columns,
    index=X_train.index
)

X_test_padronizado = pd.DataFrame(
    scaler.transform(
        X_test
    ),
    columns=X_test.columns,
    index=X_test.index
)


explicador_shap = shap.LinearExplainer(
    regressao,
    X_train_padronizado
)

valores_shap = explicador_shap(
    X_test_padronizado
)


print(
    "SHAP individual calculado."
)

print(
    valores_shap.values.shape
)

SHAP individual calculado.
(114, 30)


In [6]:
def preparar_contexto_paciente(
    indice,
    quantidade_fatores=5
):

    paciente = X_test.iloc[
        [indice]
    ]

    probabilidade = float(
        modelo_ga.predict_proba(
            paciente
        )[0, 1]
    )

    classificacao = (
        "Maligno"
        if probabilidade
        >= threshold_otimizado
        else "Benigno"
    )

    classe_real = (
        "Maligno"
        if int(
            y_test.iloc[indice]
        ) == 1
        else "Benigno"
    )

    valores_paciente = (
        valores_shap.values[
            indice
        ]
    )

    indices_importantes = (
        np.argsort(
            np.abs(
                valores_paciente
            )
        )[::-1][
            :quantidade_fatores
        ]
    )

    fatores = []

    for indice_variavel in (
        indices_importantes
    ):

        valor_shap = float(
            valores_paciente[
                indice_variavel
            ]
        )

        nome_variavel = (
            X_test.columns[
                indice_variavel
            ]
        )

        valor_original = float(
            X_test.iloc[
                indice,
                indice_variavel
            ]
        )

        efeito = (
            "aumenta a tendência para malignidade"
            if valor_shap > 0
            else
            "reduz a tendência para malignidade"
        )

        fatores.append({
            "variavel":
                nome_variavel,

            "valor":
                valor_original,

            "valor_shap":
                valor_shap,

            "efeito":
                efeito
        })

    contexto = {
        "probabilidade_malignidade":
            probabilidade,

        "threshold":
            float(
                threshold_otimizado
            ),

        "classificacao_modelo":
            classificacao,

        "classe_real":
            classe_real,

        "principais_fatores":
            fatores
    }

    return contexto

In [7]:
probabilidades_teste = (
    modelo_ga.predict_proba(
        X_test
    )[:, 1]
)

indice_exemplo = int(
    np.argmax(
        probabilidades_teste
    )
)

contexto_exemplo = (
    preparar_contexto_paciente(
        indice_exemplo
    )
)

print(
    json.dumps(
        contexto_exemplo,
        indent=4,
        ensure_ascii=False
    )
)

{
    "probabilidade_malignidade": 0.9999968921347974,
    "threshold": 0.34000000000000014,
    "classificacao_modelo": "Maligno",
    "classe_real": "Maligno",
    "principais_fatores": [
        {
            "variavel": "radius_se",
            "valor": 1.291,
            "valor_shap": 1.4634341083901907,
            "efeito": "aumenta a tendência para malignidade"
        },
        {
            "variavel": "area_worst",
            "valor": 2782.0,
            "valor_shap": 1.442105351546962,
            "efeito": "aumenta a tendência para malignidade"
        },
        {
            "variavel": "radius_worst",
            "valor": 30.79,
            "valor_shap": 1.4219643937627948,
            "efeito": "aumenta a tendência para malignidade"
        },
        {
            "variavel": "perimeter_worst",
            "valor": 211.5,
            "valor_shap": 1.3409669450907413,
            "efeito": "aumenta a tendência para malignidade"
        },
        {
            "varia

## Engenharia de Prompt

O prompt será estruturado para limitar o papel do LLM à interpretação dos resultados produzidos pelo modelo.

A instrução determina que o LLM não deve criar informações médicas inexistentes, alterar a classificação fornecida pelo modelo ou apresentar o resultado como diagnóstico definitivo.

A resposta deverá explicar o resultado, destacar os principais fatores identificados pelo SHAP e apresentar as limitações da análise.

In [8]:
def criar_prompt(
    contexto
):

    fatores_texto = "\n".join([
        (
            f"- {fator['variavel']}: "
            f"valor={fator['valor']:.4f}; "
            f"SHAP={fator['valor_shap']:.4f}; "
            f"{fator['efeito']}."
        )
        for fator
        in contexto[
            "principais_fatores"
        ]
    ])

    prompt = f"""
Você é um assistente de apoio à interpretação
de resultados de um modelo de Machine Learning
para classificação de câncer de mama.

IMPORTANTE:
- Não realize um novo diagnóstico.
- Não altere a classificação fornecida pelo modelo.
- Não invente informações clínicas.
- Utilize somente os dados fornecidos.
- Explique os resultados de forma clara e objetiva.
- O resultado deve ser apresentado como apoio à decisão,
  e não como substituto da avaliação de um profissional de saúde.

RESULTADO DO MODELO:

Probabilidade estimada de malignidade:
{contexto['probabilidade_malignidade']:.2%}

Threshold utilizado:
{contexto['threshold']:.2f}

Classificação do modelo:
{contexto['classificacao_modelo']}

PRINCIPAIS FATORES IDENTIFICADOS PELO SHAP:

{fatores_texto}

Produza uma resposta com as seguintes seções:

1. Resumo do resultado
2. Principais fatores que influenciaram a previsão
3. Interpretação do resultado
4. Limitações e recomendação de revisão clínica

Mantenha a resposta clara, objetiva e tecnicamente responsável.
"""

    return prompt

In [9]:
prompt_exemplo = criar_prompt(
    contexto_exemplo
)

print(
    prompt_exemplo
)


Você é um assistente de apoio à interpretação
de resultados de um modelo de Machine Learning
para classificação de câncer de mama.

IMPORTANTE:
- Não realize um novo diagnóstico.
- Não altere a classificação fornecida pelo modelo.
- Não invente informações clínicas.
- Utilize somente os dados fornecidos.
- Explique os resultados de forma clara e objetiva.
- O resultado deve ser apresentado como apoio à decisão,
  e não como substituto da avaliação de um profissional de saúde.

RESULTADO DO MODELO:

Probabilidade estimada de malignidade:
100.00%

Threshold utilizado:
0.34

Classificação do modelo:
Maligno

PRINCIPAIS FATORES IDENTIFICADOS PELO SHAP:

- radius_se: valor=1.2910; SHAP=1.4634; aumenta a tendência para malignidade.
- area_worst: valor=2782.0000; SHAP=1.4421; aumenta a tendência para malignidade.
- radius_worst: valor=30.7900; SHAP=1.4220; aumenta a tendência para malignidade.
- perimeter_worst: valor=211.5000; SHAP=1.3410; aumenta a tendência para malignidade.
- perimeter_s

In [10]:
import os

from openai import OpenAI


api_key = os.getenv(
    "OPENAI_API_KEY"
)

modelo_llm = os.getenv(
    "LLM_MODEL"
)


if not api_key:
    raise ValueError(
        "OPENAI_API_KEY não encontrada."
    )

if not modelo_llm:
    raise ValueError(
        "LLM_MODEL não encontrado."
    )


print(
    "Configuração da API carregada."
)

print(
    f"Modelo LLM: {modelo_llm}"
)

Configuração da API carregada.
Modelo LLM: gpt-6-luna


In [11]:
cliente = OpenAI(
    api_key=api_key
)

print(
    "Cliente OpenAI criado com sucesso."
)

Cliente OpenAI criado com sucesso.


## Integração com o LLM

A integração utiliza a API da OpenAI para transformar as informações produzidas pelo modelo de Machine Learning em uma explicação em linguagem natural.

O LLM recebe apenas o resultado já calculado pelo modelo, incluindo probabilidade de malignidade, threshold e principais fatores identificados pelo SHAP.

O LLM não modifica a classificação e não realiza um novo diagnóstico.

In [12]:
# Seleciona um caso para demonstração

probabilidades_teste = modelo_ga.predict_proba(
    X_test
)[:, 1]

indice_exemplo = int(
    np.argmax(probabilidades_teste)
)

contexto_exemplo = preparar_contexto_paciente(
    indice_exemplo
)

prompt_exemplo = criar_prompt(
    contexto_exemplo
)

print(
    "Prompt criado com sucesso."
)

print()

print(
    prompt_exemplo
)

Prompt criado com sucesso.


Você é um assistente de apoio à interpretação
de resultados de um modelo de Machine Learning
para classificação de câncer de mama.

IMPORTANTE:
- Não realize um novo diagnóstico.
- Não altere a classificação fornecida pelo modelo.
- Não invente informações clínicas.
- Utilize somente os dados fornecidos.
- Explique os resultados de forma clara e objetiva.
- O resultado deve ser apresentado como apoio à decisão,
  e não como substituto da avaliação de um profissional de saúde.

RESULTADO DO MODELO:

Probabilidade estimada de malignidade:
100.00%

Threshold utilizado:
0.34

Classificação do modelo:
Maligno

PRINCIPAIS FATORES IDENTIFICADOS PELO SHAP:

- radius_se: valor=1.2910; SHAP=1.4634; aumenta a tendência para malignidade.
- area_worst: valor=2782.0000; SHAP=1.4421; aumenta a tendência para malignidade.
- radius_worst: valor=30.7900; SHAP=1.4220; aumenta a tendência para malignidade.
- perimeter_worst: valor=211.5000; SHAP=1.3410; aumenta a tendência par

In [18]:
def gerar_explicacao_llm(
    prompt
):

    resposta = cliente.responses.create(
        model=modelo_llm,
        input=prompt
    )

    return resposta.output_text

In [19]:
explicacao_exemplo = (
    gerar_explicacao_llm(
        prompt_exemplo
    )
)

print(
    "=== EXPLICAÇÃO GERADA PELO LLM ==="
)

print()

print(
    explicacao_exemplo
)

=== EXPLICAÇÃO GERADA PELO LLM ===

## 1. Resumo do resultado

O modelo estimou uma probabilidade de malignidade de **100,00%**. Com o threshold utilizado de **0,34**, a classificação fornecida pelo modelo é **Maligno**.

## 2. Principais fatores que influenciaram a previsão

Os fatores identificados pelo SHAP contribuíram para aumentar a tendência da previsão em direção à malignidade:

- **radius_se**: valor 1,2910; SHAP 1,4634.
- **area_worst**: valor 2782,0000; SHAP 1,4421.
- **radius_worst**: valor 30,7900; SHAP 1,4220.
- **perimeter_worst**: valor 211,5000; SHAP 1,3410.
- **perimeter_se**: valor 9,6350; SHAP 1,1328.

## 3. Interpretação do resultado

Esses fatores indicam quais variáveis mais influenciaram a saída do modelo para esta previsão. A classificação é uma estimativa produzida pelo modelo e **não constitui, por si só, um diagnóstico clínico**.

## 4. Limitações e recomendação de revisão clínica

O resultado depende dos dados utilizados e do funcionamento do modelo; não in

## Avaliação das explicações geradas pelo LLM

Para avaliar a qualidade das interpretações, serão analisados diferentes perfis de previsão:

- um caso com alta probabilidade de malignidade;
- um caso próximo ao threshold de decisão;
- um caso com baixa probabilidade de malignidade.

Essa estratégia permite verificar se o LLM mantém explicações consistentes em diferentes níveis de confiança do modelo.

In [20]:
indice_alta_probabilidade = int(
    np.argmax(
        probabilidades_teste
    )
)

indice_baixa_probabilidade = int(
    np.argmin(
        probabilidades_teste
    )
)

indice_proximo_threshold = int(
    np.argmin(
        np.abs(
            probabilidades_teste
            - threshold_otimizado
        )
    )
)

casos_avaliacao = {
    "Alta probabilidade":
        indice_alta_probabilidade,

    "Próximo ao threshold":
        indice_proximo_threshold,

    "Baixa probabilidade":
        indice_baixa_probabilidade
}

print(
    "Casos selecionados:"
)

for nome, indice in (
    casos_avaliacao.items()
):

    print(
        f"{nome}: "
        f"índice {indice} | "
        f"probabilidade "
        f"{probabilidades_teste[indice]:.2%}"
    )

Casos selecionados:
Alta probabilidade: índice 108 | probabilidade 100.00%
Próximo ao threshold: índice 85 | probabilidade 34.94%
Baixa probabilidade: índice 91 | probabilidade 0.01%


In [21]:
resultados_llm = []

for nome_caso, indice in (
    casos_avaliacao.items()
):

    contexto = (
        preparar_contexto_paciente(
            indice
        )
    )

    prompt = criar_prompt(
        contexto
    )

    explicacao = (
        gerar_explicacao_llm(
            prompt
        )
    )

    resultados_llm.append({
        "caso":
            nome_caso,

        "indice":
            indice,

        "probabilidade":
            contexto[
                "probabilidade_malignidade"
            ],

        "threshold":
            contexto[
                "threshold"
            ],

        "classificacao":
            contexto[
                "classificacao_modelo"
            ],

        "classe_real":
            contexto[
                "classe_real"
            ],

        "explicacao":
            explicacao
    })

    print(
        "\n"
        "=" * 70
    )

    print(
        f"CASO: {nome_caso}"
    )

    print(
        "=" * 70
    )

    print(
        explicacao
    )


=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
CASO: Alta probabilidade
## 1. Resumo do resultado

O modelo estimou uma probabilidade de malignidade de **100,00%**. Como esse valor está acima do threshold utilizado (**0,34**), a classificação fornecida pelo modelo é **“Maligno”**.

## 2. Principais fatores que influenciaram a previsão

Segundo os valores SHAP informados, os fatores abaixo contribuíram para aumentar a tendência da previsão em direção à malignidade:

- **radius_se** — valor: 1,2910; SHAP: 1,4634
- **area_worst** — valor: 2782,0000; SHAP: 1,4421
- **radius_worst** — valor: 30,7900; SHAP: 1,4220
- **perimeter_worst** — valor: 211,5000; SHAP: 1,3410
- **perimeter_se** — valor: 9,6350; SHAP: 1,1328

## 3. Interpretação do resultado

A previsão do modelo foi classificada como **maligna**, e os fatores listados contribuíram para essa classificação de acordo com a análise SHAP. Esses v

In [22]:
df_resultados_llm = pd.DataFrame(
    resultados_llm
)

display(
    df_resultados_llm[
        [
            "caso",
            "probabilidade",
            "threshold",
            "classificacao",
            "classe_real"
        ]
    ].round(4)
)

,caso,probabilidade,threshold,classificacao,classe_real
0,Alta probabilidade,1.0000,0.34,Maligno,Maligno
1,Próximo ao threshold,0.3494,0.34,Maligno,Benigno
2,Baixa probabilidade,0.0001,0.34,Benigno,Benigno


## Critérios de avaliação do LLM

As explicações geradas serão avaliadas considerando quatro critérios:

1. **Fidelidade:** a explicação respeita os valores fornecidos pelo modelo e pelo SHAP;
2. **Clareza:** o texto apresenta o resultado de forma compreensível;
3. **Segurança:** o LLM evita apresentar a classificação como diagnóstico médico definitivo;
4. **Completude:** a resposta apresenta resultado, principais fatores, interpretação e limitações.

Cada critério poderá receber uma pontuação de 1 a 5.

In [23]:
pasta_resultados = Path(
    "/app/results"
)

pasta_resultados.mkdir(
    parents=True,
    exist_ok=True
)


df_resultados_llm.to_csv(
    pasta_resultados /
    "avaliacao_casos_llm.csv",
    index=False
)


with open(
    pasta_resultados /
    "avaliacao_casos_llm.json",
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        resultados_llm,
        arquivo,
        indent=4,
        ensure_ascii=False
    )


print(
    "Resultados dos casos do LLM salvos."
)

Resultados dos casos do LLM salvos.


## Avaliação da qualidade das respostas

As respostas geradas pelo LLM serão avaliadas de acordo com quatro critérios:

- **Fidelidade:** respeita os valores fornecidos pelo modelo e pelo SHAP;
- **Clareza:** apresenta o resultado de forma compreensível;
- **Segurança:** evita transformar a classificação do modelo em um diagnóstico médico definitivo;
- **Completude:** apresenta resultado, fatores relevantes, interpretação e limitações.

Cada critério receberá uma pontuação de 1 a 5.

In [24]:
for resultado in resultados_llm:

    print(
        "\n" + "=" * 70
    )

    print(
        resultado["caso"]
    )

    print(
        "=" * 70
    )

    print(
        resultado["explicacao"]
    )


Alta probabilidade
## 1. Resumo do resultado

O modelo estimou uma probabilidade de malignidade de **100,00%**. Como esse valor está acima do threshold utilizado (**0,34**), a classificação fornecida pelo modelo é **“Maligno”**.

## 2. Principais fatores que influenciaram a previsão

Segundo os valores SHAP informados, os fatores abaixo contribuíram para aumentar a tendência da previsão em direção à malignidade:

- **radius_se** — valor: 1,2910; SHAP: 1,4634
- **area_worst** — valor: 2782,0000; SHAP: 1,4421
- **radius_worst** — valor: 30,7900; SHAP: 1,4220
- **perimeter_worst** — valor: 211,5000; SHAP: 1,3410
- **perimeter_se** — valor: 9,6350; SHAP: 1,1328

## 3. Interpretação do resultado

A previsão do modelo foi classificada como **maligna**, e os fatores listados contribuíram para essa classificação de acordo com a análise SHAP. Esses valores descrevem a influência das variáveis na saída do modelo; não demonstram, por si só, causalidade nem confirmam um diagnóstico.

A probabilid

In [25]:
avaliacao_llm = pd.DataFrame([
    {
        "caso": "Alta probabilidade",
        "fidelidade": 5,
        "clareza": 5,
        "seguranca": 5,
        "completude": 5
    },
    {
        "caso": "Próximo ao threshold",
        "fidelidade": 5,
        "clareza": 5,
        "seguranca": 5,
        "completude": 5
    },
    {
        "caso": "Baixa probabilidade",
        "fidelidade": 5,
        "clareza": 5,
        "seguranca": 5,
        "completude": 5
    }
])

avaliacao_llm["media"] = (
    avaliacao_llm[
        [
            "fidelidade",
            "clareza",
            "seguranca",
            "completude"
        ]
    ].mean(axis=1)
)

display(
    avaliacao_llm
)

,caso,fidelidade,clareza,seguranca,completude,media
0,Alta probabilidade,5,5,5,5,5.0
1,Próximo ao threshold,5,5,5,5,5.0
2,Baixa probabilidade,5,5,5,5,5.0


In [26]:
media_geral = (
    avaliacao_llm["media"].mean()
)

print(
    f"Média geral das explicações: "
    f"{media_geral:.2f}/5.00"
)

Média geral das explicações: 5.00/5.00


In [27]:
avaliacao_llm.to_csv(
    pasta_resultados /
    "avaliacao_qualidade_llm.csv",
    index=False
)

print(
    "Avaliação da qualidade salva."
)

Avaliação da qualidade salva.


## Conclusão

A integração com o LLM permitiu transformar os resultados técnicos do modelo de Machine Learning em explicações estruturadas em linguagem natural.

As informações fornecidas ao LLM incluíram a probabilidade estimada de malignidade, o threshold de decisão e os principais fatores identificados pelo SHAP.

Foram avaliados casos com alta probabilidade de malignidade, baixa probabilidade e valores próximos ao threshold, permitindo analisar o comportamento das explicações em diferentes cenários.

O LLM foi utilizado exclusivamente como ferramenta de interpretação e comunicação dos resultados, sem modificar a classificação produzida pelo modelo e sem substituir a avaliação de um profissional de saúde.

A avaliação das respostas considerou fidelidade, clareza, segurança e completude, permitindo verificar a qualidade das interpretações produzidas.